In [ ]:
# new_get_wiki_data.py
#
# Description: This script retrieves and processes mass spectrometry data from MassWiki.
#              It includes functions for filtering results and querying spectral library hits.
#
# Author: Adapted from zyang2k's R script
# Last Modified: 2025-01-23

#------------------------------------------------------------------------------
# Required Packages
#------------------------------------------------------------------------------
import pandas as pd  # For data manipulation
import requests  # For HTTP requests
import json  # For JSON parsing
import urllib.parse  # For URL encoding

#------------------------------------------------------------------------------
# Function Definitions
#------------------------------------------------------------------------------

def filter_masswiki_results(masswiki_result):
    """
    Filter mass spectrometry results based on specific criteria:
    1. Keeps only manually annotated entries
    2. Removes entries with names starting with 'yy' or 'zz'

    Args:
        masswiki_result (pd.DataFrame): A DataFrame containing MassWiki results
            with columns including 'is_manual_annotated' and 'name'

    Returns:
        pd.DataFrame: Filtered results containing only manually annotated entries
            with valid names
    
    Examples:
        filtered_data = filter_masswiki_results(raw_results)
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = masswiki_result.copy()
    
    # Convert 'name' column to string, replacing NaNs with empty string
    df['name'] = df['name'].astype(str).fillna('')
    
    # Ensure 'is_manual_annotated' is boolean, replacing NaNs with False
    df['is_manual_annotated'] = df['is_manual_annotated'].fillna(False).astype(bool)
    
    # Filter manually annotated entries and exclude names starting with 'yy' or 'zz'
    filtered_result = df[
        (df['is_manual_annotated'] == True) & 
        (~df['name'].str.lower().str.startswith(('yy', 'zz')))
    ]
    
    return filtered_result

def get_spectrum_data(wiki_ids, bearer_token=None):
    """
    Retrieve spectral data for given wiki IDs from the MassWiki API.

    The function handles multiple wiki IDs and includes error handling for API requests.

    Args:
        wiki_ids (list): A list of wiki IDs to query
        bearer_token (str): Bearer token for API authentication

    Returns:
        dict: A nested dictionary where each key is a wiki_id and contains:
            - reference_library: Results from reference library identity search
            - annotation_library: Results from annotation library identity search
            Returns None for entries where the API request failed

    Details:
    1. Processes each wiki_id individually
    2. Skips empty or None wiki_ids
    3. Uses requests params dict (requests handles URL encoding automatically)
    4. Makes GET requests to the MassWiki API with Bearer token authentication
    5. Parses both reference and annotation library results

    Examples:
        results = get_spectrum_data(['wiki_id_1', 'wiki_id_2'], bearer_token='your_token')
    """
    # Ensure wiki_ids is a list and remove any None or empty string values
    wiki_ids = [str(wiki_id) for wiki_id in wiki_ids if wiki_id]
    
    # Initialize results dictionary
    all_results = {}
    
    # Set up headers with Bearer token authentication
    headers = {'Accept': 'application/json'}
    if bearer_token:
        headers['Authorization'] = f'Bearer {bearer_token}'
    
    # Base API URL
    base_url = 'https://masswiki.us-west-2.elasticbeanstalk.com/analysis/get_data'
    
    # Process each wiki_id
    for wiki_id in wiki_ids:
        try:
            # Prepare query parameters (requests will handle URL encoding automatically)
            params = {
                'wiki_id': wiki_id,      # DO NOT pre-encode - requests handles it
                'source': 'binbase',     # required parameter
                'isPublic': 'false',     # required parameter
            }
            
            # Make the API request with query params and authentication headers
            response = requests.get(base_url, params=params, headers=headers, timeout=20)
            
            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                
                results = {}
                
                # Extract reference library identity search results if they exist
                if data.get('analysis', {}).get('reference_library', {}).get('identity_search'):
                    results['reference_library'] = data['analysis']['reference_library']['identity_search']
                else:
                    results['reference_library'] = None
                
                # Extract annotation library identity search results if they exist
                if data.get('analysis', {}).get('annotation_library', {}).get('identity_search'):
                    results['annotation_library'] = data['analysis']['annotation_library']['identity_search']
                else:
                    results['annotation_library'] = None
                
                all_results[wiki_id] = results
                
            else:
                print(f"Failed to get spectrum data for {wiki_id}. Status code: {response.status_code}")
                all_results[wiki_id] = None
        
        except Exception as e:
            print(f"Error processing wiki_id: {wiki_id} - {str(e)}")
            all_results[wiki_id] = None
    
    return all_results

#------------------------------------------------------------------------------
# Usage Notes:
# 1. Ensure all required packages are installed:
#    pip install pandas requests
# 2. Verify that input CSV files exist in the data/ directory
# 3. Check internet connection for API requests
# 4. Consider implementing rate limiting for large numbers of API requests
# 5. Monitor API response times and implement appropriate timeout settings
# 6. Provide a valid Bearer token for API authentication
# 7. Required parameters: wiki_id, source='binbase', isPublic='false'
#------------------------------------------------------------------------------

# Example usage (uncomment and modify as needed)
# def main():
#     # Read input data
#     raw_data = pd.read_csv('data/input_file.csv')
#     
#     # Filter results
#     filtered_data = filter_masswiki_results(raw_data)
#     
#     # Get spectrum data with authentication
#     bearer_token = "your_token_here"
#     spectrum_results = get_spectrum_data(filtered_data['wiki_id'].tolist(), bearer_token=bearer_token)
#     
#     # Further processing can be done here
# 
# if __name__ == '__main__':
#     main()


In [11]:
wiki = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/data/hilic_orbi_neg.csv')


In [12]:
filtered_wiki = filter_masswiki_results(wiki)


In [13]:
filtered_wiki.shape

(1771, 43)

In [ ]:
bearer_token = "eyJraWQiOiJoeCtPbm1BUmpUWG0rZnNzZnptYVR2b2RIeG1Ra0dKbGVzc1hsZG5oTG5nPSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiI4ODYxYzM2MC1lMGExLTcwOTMtM2JjNC0wZDkxZDlmOGFkN2UiLCJjb2duaXRvOmdyb3VwcyI6WyJtYXNzd2lraS1sYWIiLCJVc2VycyJdLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiaXNzIjoiaHR0cHM6XC9cL2NvZ25pdG8taWRwLnVzLXdlc3QtMi5hbWF6b25hd3MuY29tXC91cy13ZXN0LTJfR2p0Y00wUENwIiwiY29nbml0bzp1c2VybmFtZSI6Ijg4NjFjMzYwLWUwYTEtNzA5My0zYmM0LTBkOTFkOWY4YWQ3ZSIsIm9yaWdpbl9qdGkiOiJmNWE3YjQzMS1hYjMyLTQ0ZTgtOGRiOC0zZGVmMWJlMGVmZjUiLCJhdWQiOiIzaGdvczNhdGQxZWwxNmx0aDYxN2lpY29hbCIsImV2ZW50X2lkIjoiNGFmMTRmZTUtZGY5OS00MDI3LTk1NjktOTQ0YmI1ZDU5YWFhIiwidG9rZW5fdXNlIjoiaWQiLCJhdXRoX3RpbWUiOjE3NzM3MDgzMzUsIm5hbWUiOiJaaXl1ZSBZYW5nIiwiZXhwIjoxNzczNzc3NDg0LCJpYXQiOjE3NzM3NzM4ODQsImZhbWlseV9uYW1lIjoiWWFuZyIsImp0aSI6IjhhMzE4NDg5LWU4ODYtNDQwMC05Yjc0LWNjYzBkNjlmNGRlYSIsImVtYWlsIjoienl6eWFuZ0B1Y2RhdmlzLmVkdSJ9.VDLOVvC7ZWUHX5rwGOgB7wwYWdEj0jLhxuvBZLrEKhXrl2Tk-uw8c6ro13h-ODkFqeM7Q1915fLbHDdF2AKDfRnT817HGIfJle_YS8VPKv5JqPy8NmFPnlC_yOVbFfhkGmOki5hhke6nhv02uq0CDko65ZwcjUrStQZbLmjgWFzyQWbI8w37KO_uc2bHJde-Yr1l90O2DnqmsmjTMVtWLdYbEXH5v566dP5K51QIjrRUIDA1USsCcfB86WOcTx9akIr5Mr4kad4c-GHsHD0knEi2ZbweChQO_uqTsSFQidSYH0edieJb6F_9GhYXJ_gBsIhEyIQwyFrqKhZrzlCiRQ"

wiki_ids = filtered_wiki['wiki_id'].tolist()

print(f"Fetching data for {len(wiki_ids)} wiki IDs...")
# Get spectrum data for all IDs with authentication
wiki_matches = get_spectrum_data(wiki_ids, bearer_token=bearer_token)
print(f"Done! Retrieved data for {len([v for v in wiki_matches.values() if v is not None])} specs")


Failed to get spectrum data for aPUDE1U/22. Status code: 400
Failed to get spectrum data for aPUDE1U/143. Status code: 400
Failed to get spectrum data for aPUDE1U/146. Status code: 400
Failed to get spectrum data for aPUDE1U/154. Status code: 400
Failed to get spectrum data for aPUDE1U/199. Status code: 400
Failed to get spectrum data for aPUDE1U/200. Status code: 400
Failed to get spectrum data for aPUDE1U/202. Status code: 400
Failed to get spectrum data for aPUDE1U/208. Status code: 400
Failed to get spectrum data for aPUDE1U/209. Status code: 400


In [ ]:
for wiki_id, libraries in wiki_matches.items():
    ref_results = libraries['reference_library']
    annotation_results = libraries['annotation_library']
    
    # Process or examine results for each wiki_id

In [16]:
# Debug: Test single wiki_id with correct params format
test_wiki_id = wiki_ids[0]
print(f"Testing wiki_id: {test_wiki_id}")

headers = {
    'Accept': 'application/json',
    'Authorization': f'Bearer {bearer_token}'
}

# Use params dict instead of building URL manually
# This is the correct way to call MassWiki API
params = {
    'wiki_id': test_wiki_id,  # DO NOT pre-encode - requests handles it
    'source': 'binbase',       # required
    'isPublic': 'false',       # required
}

url = 'https://masswiki.us-west-2.elasticbeanstalk.com/analysis/get_data'
response = requests.get(url, params=params, headers=headers, timeout=20)

print(f"URL sent (by requests): {response.url}")
print(f"Status: {response.status_code}")
print(f"Response: {response.text[:500]}")

if response.status_code == 200:
    data = response.json()
    print("\nSuccess! JSON structure:")
    print(f"  Keys: {data.keys() if isinstance(data, dict) else 'N/A'}")


Testing wiki_id: aPUDE1U/6
URL sent (by requests): https://masswiki.us-west-2.elasticbeanstalk.com/analysis/get_data?wiki_id=aPUDE1U%2F6&source=binbase&isPublic=false
Status: 200
Response: {"analysis":{"reference_library":{"identity_search":[{"db":"NIST23","id":"290689","name":"Val-Tyr-Val","adduct":"[M-H]-","rt":"","precursor_mz":"378.2034","smiles":"CC(C)C(N)C(=O)NC(Cc1ccc(O)cc1)C(=O)NC(C(=O)O)C(C)C","user_name":"","spec_uid":"","predicted_rt_hilic":"99.8","predicted_rt_rp":"12.1","anno_delta_rt":"","index":269464,"entropy_similarity":0.9709408283233643,"library_wiki_id":"nist23/-1/269464","delta_predicted_rt":-0.7},{"db":"NIST23","id":"290690","name":"Val-Tyr-Val","adduct":"[M-

Success! JSON structure:
  Keys: dict_keys(['analysis', 'spectrum', 'wiki_id', 'status'])


# Unrelateed

In [ ]:
import pickle
import numpy as np


# Load the pickle file
with open('/Users/ellayoung/Desktop/metabolo_confi_score/data/NIST23_negative_entropy_selected_adducts.pkl', 'rb') as f:
    nist_data = pickle.load(f)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

def extract_features(entry, wiki_matches=None):
    # Default features from entry
    features = {
        'exact_mass': entry.get('ExactMass', entry.get('exact_mass', 0)),
        'precursor_mz': entry.get('precursor_mz', entry.get('Precursor_mz', 0)),
        'fragment_count': entry.get('Num Peaks', entry.get('fragment_count', 0)),
        'entropy_similarity': 0,
        'rt_available': 0
    }
    
    # Optional: Try to enrich features from wiki_matches if provided
    if wiki_matches:
        for libraries in wiki_matches.values():
            ref_results = libraries.get('reference_library', [])
            if ref_results:  # Check if ref_results is not None and not empty
                for result in ref_results:
                    if isinstance(result, dict):
                        features['entropy_similarity'] = result.get('entropy_similarity', 0)
                        features['rt_available'] = 1 if result.get('rt', '') else 0
                        break
    
    return features

# Main dataset preparation function
def prepare_classification_dataset(nist_data, wiki_matches=None):
    X, y = [], []
    
    # Process NIST data as positive examples
    for nist_entry in list(nist_data):
        features = extract_features(nist_entry, wiki_matches)
        X.append(list(features.values()))
        y.append(1)  # Positive class
    
    # Optional: Process wiki matches as potential negative examples
    if wiki_matches:
        for libraries in wiki_matches.values():
            ref_results = libraries.get('reference_library', [])
            if ref_results:
                for result in ref_results:
                    features = extract_features(result)
                    X.append(list(features.values()))
                    y.append(0)  # Negative class
    
    return np.array(X), np.array(y)

def create_metabolite_classifier(X, y):
    """
    Build and train logistic regression classifier
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    
    # Preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), [0, 1, 2, 3]),  # Numeric columns
            ('cat', OneHotEncoder(handle_unknown='ignore'), [4, 5])  # Categorical columns
        ])
    
    # Full pipeline
    classifier = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000))
    ])
    
    # Train classifier
    classifier.fit(X_train, y_train)
    
    return classifier

def calculate_identification_confidence(entry, classifier):
    """
    Calculate confidence score for a metabolite entry
    """
    features = extract_features(entry)
    confidence_score = classifier.predict_proba([list(features.values())])[0][1]
    return confidence_score

In [ ]:
# Assuming nist_data and wiki_matches are already defined in your environment
X, y = prepare_classification_dataset(nist_data, wiki_matches)
classifier = create_metabolite_classifier(X, y)

# Optional: Calculate confidence for a specific entry
# entry = some_metabolite_entry
# confidence = calculate_identification_confidence(entry, classifier)